In [ ]:
import pandas as pd
import numpy as np
from ChromaVDB.chroma import ChromaFramework
from DeepGraphDB import DeepGraphDB
from tqdm.notebook import tqdm
import torch
import pickle

gdb = DeepGraphDB()
gdb.load_graph("/home/cc/PHD/dglframework/DeepKG/DeepGraphDB/graphs/primekg.bin")

vdb = ChromaFramework(persist_directory="./ChromaVDB/chroma_db")
records = vdb.list_records()

# names = [record['name'] for record in records if record['embedding_type'] == 'graph']
# entities = [record['entity'] for record in records if record['embedding_type'] == 'graph']
graph_embs = [record['embeddings'] for record in records if record['embedding_type'] == 'graph']
# text_embs = [record['embeddings'] for record in records if record['embedding_type'] == 'text']
ids = [record['id'] for record in records if record['embedding_type'] == 'graph']

data = pd.read_excel('data/2025_03_29.xlsx') # Provare ad usare anche stadio-avanzato, IPI e Log10hGE

with open("/home/cc/PHD/dglframework/DeepKG/conditioning.pkl", "rb") as f:
    conditioning = pickle.load(f)

target = 35884

In [ ]:
id_to_original_index = {v: k for k, v in vdb.global_to_vids_mapping.items()}

global_ids = [id_to_original_index[id] for id in ids]

start_feats = [graph_embs[i] for i in global_ids]

gdb.load_node_features_for_gnn(torch.tensor(start_feats, dtype=torch.float32))

In [ ]:
nodes_to_keep = {}
scores_dict = {}

conditioning.append({ 'entity': 'disease: diffuse large B-cell lymphoma', 'score': 5.0 })

for item in conditioning:
    entity = item['entity'].split(': ')
    score = item['score']

    if entity[1] in gdb.node_data[entity[0]]['name']:
        idx = np.where(gdb.node_data[entity[0]]['name'] == entity[1])[0][0]

        nodes_to_keep[entity[0]] = nodes_to_keep.get(entity[0], []) + [idx]
        scores_dict[entity[0]] = scores_dict.get(entity[0], []) + [score]
    # else:
    #     print(f"Entity {entity[1]} not found in graph for type {entity[0]}")

for entity_type, scores in scores_dict.items():
    scores_tensor = torch.tensor(scores, dtype=torch.float32, device='cuda')

    normalized_scores = scores_tensor / 5.0
    scores_dict[entity_type] = normalized_scores

finetune_graph = gdb.graph.subgraph(nodes_to_keep)

gdb.graph = finetune_graph

In [ ]:
import torch
import dgl
import numpy as np
from collections import defaultdict
from sklearn.model_selection import train_test_split
from DeepGraphDB.gnns.heteroSAGEattn import AdvancedHeteroLinkPredictor, compute_loss

in_feats = {ntype: gdb.graph.nodes['disease'].data['x'].shape[1] for ntype in gdb.graph.ntypes}

# Choose multiple edge types for prediction
target_etypes = [ctype for ctype in gdb.graph.canonical_etypes if (ctype[0] == "geneprotein" or ctype[2] in "geneprotein") and gdb.graph.num_edges(ctype) > 10000]
# target_etypes = [ctype for ctype in gdb.graph.canonical_etypes if (ctype[0] == "geneprotein" or ctype[2] in "geneprotein")]

# target_etypes = [('geneprotein', 'protein_protein', 'geneprotein'), ('disease', 'disease_protein', 'geneprotein'),  ('geneprotein', 'disease_protein', 'disease')]

print(f"Target edge types for prediction: {target_etypes}")

hidden_feats = 512
out_feats = 512

model = AdvancedHeteroLinkPredictor(
    node_types=gdb.graph.ntypes,  # All node types in the graph
    edge_types=gdb.graph.etypes,  # All edge types for GNN layers
    canonical_etypes=gdb.graph.canonical_etypes,  # All canonical edge types,
    in_feats=in_feats,
    hidden_feats=hidden_feats,
    out_feats=out_feats,
    # scores=scores_dict,  # Use scores for link prediction
    scores=None,  
    num_layers=3,
    use_attention=True,
    predictor_type='mlp',
    target_etypes=target_etypes,  # Only target edge types for prediction
    gnn_type='rgcn'
    # gnn_type="sage"
)

print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")

embs = gdb.train_model(model, compute_loss, target_etypes, 'cuda', bs=1000000, num_epochs=170)

In [ ]:
torch.save(model, "/home/cc/PHD/dglframework/DeepKG/models/model-SAGE-bcell-finetune.pt")

for entity in gdb.graph.ntypes:
    embeddings_tensor = embs[entity].cpu()
    torch.save(embeddings_tensor, f"/home/cc/PHD/dglframework/DeepKG/finetune-embs/{entity}.pt")

In [ ]:
gene_embs = torch.load("/home/cc/PHD/dglframework/DeepKG/finetune-embs/geneprotein.pt")
disease_embs = torch.load("/home/cc/PHD/dglframework/DeepKG/finetune-embs/disease.pt")

gene_names = [ gdb.node_data['geneprotein']['name'][lid] for lid in nodes_to_keep['geneprotein'] ]
disease_names = [ gdb.node_data['disease']['name'][lid] for lid in nodes_to_keep['disease'] ]

# gene_set = "LNF"
gene_set = "plasma"
# gene_measure = "MUT"
gene_measure = "VAF"

gene_data = data[[col for col in data.columns if gene_set in col and gene_measure in col]]

genes = list(set([ gene.split('_')[0] for gene in gene_data.columns ]))

final_columns = []
embeddings = []
record_ids = []

for gene in genes:
    if gene in gene_names:
        final_columns.append(gene+"_"+gene_set+"_"+gene_measure)
        embeddings.append(gene_embs[gene_names.index(gene)])
        record_ids.append(gene_names.index(gene))
    else:
        print(gene)

print(len(final_columns))

gene_data = gene_data[final_columns]
gene_data['pfs'] = data['PFS_Cens_updated']
# gene_data = gene_data.dropna()
gene_data = gene_data.fillna(0)

In [ ]:
import torch.nn as nn

patient_mutations = torch.tensor(gene_data.drop(columns=['pfs']).values, dtype=torch.float64)
gene_embeddings = torch.tensor(np.array(embeddings))

# Weight gene embeddings by mutation status
mutations_expanded = patient_mutations.unsqueeze(-1)
gene_weight = nn.Linear(1, 1, bias=False, dtype=torch.float64)
weighted_mutations = gene_weight(mutations_expanded)

gene_emb_expanded = gene_embeddings.unsqueeze(0)
weighted_gene_embs = weighted_mutations * gene_emb_expanded

# Sum across genes
patient_embs = weighted_gene_embs.sum(dim=1).detach()
labels = torch.tensor(gene_data['pfs'].values, dtype=torch.float64)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import torch

def visualize_embeddings_tsne(embeddings, labels, perplexity=30, n_iter=1000, random_state=42):
    """
    Visualize tensor embeddings using t-SNE with binary labels
    
    Args:
        embeddings: List of tensors or numpy arrays, or a single tensor/array
        labels: List of binary labels (0s and 1s)
        perplexity: t-SNE perplexity parameter (default: 30)
        n_iter: Number of iterations for t-SNE (default: 1000)
        random_state: Random state for reproducibility (default: 42)
    """
    
    # Convert tensors to numpy if needed
    if isinstance(embeddings, list):
        if torch.is_tensor(embeddings[0]):
            # Convert list of tensors to numpy array
            embs_np = torch.stack(embeddings).detach().cpu().numpy()
        else:
            # Assume list of numpy arrays
            embs_np = np.array(embeddings)
    elif torch.is_tensor(embeddings):
        # Single tensor
        embs_np = embeddings.detach().cpu().numpy()
    else:
        # Assume numpy array
        embs_np = embeddings
    
    # Reshape if needed (flatten each embedding)
    if len(embs_np.shape) > 2:
        embs_np = embs_np.reshape(embs_np.shape[0], -1)
    
    # Convert labels to numpy array
    labels_np = np.array(labels)
    
    print(f"Embedding shape: {embs_np.shape}")
    print(f"Labels shape: {labels_np.shape}")
    print(f"Unique labels: {np.unique(labels_np)}")
    
    # Apply t-SNE
    print("Applying t-SNE...")
    tsne = TSNE(n_components=2, init='pca', perplexity=perplexity, n_iter=n_iter, random_state=random_state)
    # tsne = PCA(n_components=2)
    embeddings_2d = tsne.fit_transform(embs_np)
    
    # Create the plot
    plt.figure(figsize=(10, 8))
    
    # Plot points with different colors for different labels
    colors = ['red', 'blue']
    labels_text = ['Label 0', 'Label 1']
    
    for i, label in enumerate([0, 1]):
        mask = labels_np == label
        if np.any(mask):  # Only plot if this label exists
            plt.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1], 
                       c=colors[i], label=labels_text[i], alpha=0.7, s=50)
    
    plt.title('t-SNE Visualization of Embeddings', fontsize=16)
    plt.xlabel('t-SNE Component 1', fontsize=12)
    plt.ylabel('t-SNE Component 2', fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Show the plot
    plt.show()
    
    return embeddings_2d, tsne

embeddings_2d, tsne_model = visualize_embeddings_tsne(patient_embs, labels, perplexity=15)

print(f"Final 2D embeddings shape: {embeddings_2d.shape}")

In [ ]:
# dfstring = '_mutated'
dfstring = '_plasma_MUT'

mutation_counts = data.filter(like=dfstring).sum()

mutation_counts = mutation_counts.sort_values(ascending=False)
# 2. Filter for genes with at least 1 mutation
genes_with_mutations = mutation_counts[mutation_counts >= 1]

# 3. Calculate the mean of this filtered group
mean_mutation_count = genes_with_mutations.mean()

print("--- Genes With at Least 1 Mutation ---")
print(len(genes_with_mutations))

print(f"\n--- Mean Number of Mutations ---")
print(mean_mutation_count)

In [ ]:
genes_with_mutations

In [ ]:
import torch.nn.functional as F
# Define how many top results you want to see for each disease
TOP_K = 10

diseases = {name: index for index, name in enumerate(disease_names) if "diffuse large b-cell lymphoma" in name.lower()}

# 1. Filter for all 'geneproteint' embeddings and their names
print("Filtering for gene/protein embeddings...")


# 2. Iterate through your target diseases and find similar embeddings
print("\n--- Finding Most Similar Gene/Proteins ---")
for disease_name, disease_index in diseases.items():
    print(f"\nDisease: {disease_name}")

    # Get the embedding for the current disease and convert it to a tensor
    disease_emb = torch.tensor(disease_embs[disease_index], dtype=torch.float32)

    # Calculate cosine similarity between the disease and ALL gene/protein embeddings
    # We use unsqueeze(0) to make the disease_emb 2D for broadcasting [1, D] vs [N, D]
    similarities = F.cosine_similarity(disease_emb.unsqueeze(0), gene_embs)

    # Get the top K results (both values and their indices)
    top_results = torch.topk(similarities, k=TOP_K)
    
    # 3. Display the results
    for i in range(TOP_K):
        score = top_results.values[i].item()
        gene_index = top_results.indices[i].item()
        gene_name = gene_names[gene_index]
        print(f"  {i+1}. {gene_name} (Similarity: {score:.4f}) - mutation count: {genes_with_mutations.get(gene_name+dfstring, 0)}")